In [1]:
# add imports
import gemcat as gc
import numpy as np
import pandas as pd
import glob

import warnings
warnings.filterwarnings('ignore')

The genome scale metabolic model of rat was taken from Wang et al., 2021 (https://www.pnas.org/doi/full/10.1073/pnas.2102344118)

In [2]:
import cobra
rat_filepath = '../data/rat/model/'
rgenes = pd.read_csv(rat_filepath+'GENES_RAT.tsv', sep='\t')
rmodel = cobra.io.read_sbml_model(rat_filepath+'Rat-GEM.xml')
rmodel

https://identifiers.org/taxonomy/ does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id


Name,RatGEM
Memory address,17baf2a10
Number of metabolites,8458
Number of reactions,12995
Number of genes,2804
Number of groups,150
Objective expression,1.0*MAR00021 - 1.0*MAR00021_reverse_97974
Compartments,"Cytosol, Extracellular, Lysosome, Endoplasmic reticulum, Mitochondria, Peroxisome, Golgi apparatus, Nucleus, Inner mitochondria"


In [3]:
# Split the 'ENSEMBL_ID' column by ';' and create new rows for each value
rgenes_exploded = rgenes.assign(ENSEMBL_ID=rgenes['ENSEMBL_ID'].str.split(';')).explode('ENSEMBL_ID')

# Split the "UNIPROT_ID" column by ';' and create new rows for each value
rgenes_exploded = rgenes_exploded.assign(UNIPROT_ID=rgenes_exploded['UNIPROT_ID'].str.split(';')).explode('UNIPROT_ID')

# Reset the index if needed
rgenes_exploded.reset_index(drop=True, inplace=True)
rgenes_new = rgenes_exploded[(rgenes_exploded['ENSEMBL_ID'].notna()) & (rgenes_exploded['UNIPROT_ID'].notna())]

# get the common genes
rgenes_model = [g.id for g in rmodel.genes]
common_genes = rgenes_new[rgenes_new['SYMBOL'].isin(rgenes_model)]


## Analyses of the proteomics

In [4]:
print('Number of samples in each tissue:')
for file in glob.glob('../data/rat/proteomics/*csv'):
    _tissue = file.split('_')[-1].split('.')[0]
    _df = pd.read_csv(file)
    print(f'\t{_tissue}: {len([i for i in _df.columns if i.isdigit()])}')

Number of samples in each tissue:
	t55-gastrocnemius: 60
	t68-liver: 60
	t58-heart: 60
	t70-white-adipose: 60
	t53-cortex: 60
	t59-kidney: 60
	t66-lung: 60


## Analyses of the transcriptomics

In [5]:
print('Number of samples in each tissue:')
for file in glob.glob('../data/rat/rnaseq/*csv'):
    _tissue = file.split('_')[-1].split('.')[0]
    _df = pd.read_csv(file)
    print(f'\t{_tissue}: {len([i for i in _df.columns if i.isdigit()])}')

Number of samples in each tissue:
	t68-liver: 54
	t67-small-intestine: 50
	t58-heart: 52
	t66-lung: 52
	t59-kidney: 52
	t53-cortex: 50
	t70-white-adipose: 52
	t69-brown-adipose: 52
	t64-ovaries: 24
	t61-colon: 50
	t52-hippocampus: 56
	t63-testes: 25
	t62-spleen: 50
	t56-vastus-lateralis: 50
	t60-adrenal: 52
	t55-gastrocnemius: 62
	t99-vena-cava: 50
	t54-hypothalamus: 50


## Calculation of PR centrality using GEMCAT

In [36]:
tissue = 't59-kidney'

# load the proteomics data
df_proteomics = pd.read_csv([i for i in glob.glob('../data/rat/proteomics/*.csv') if tissue in i][0], index_col=0)

# load the transcriptomics data
df_transcriptomics = pd.read_csv([i for i in glob.glob('../data/rat/rnaseq/*.csv') if tissue in i][0], index_col=0)

### Proteomics

In [35]:
df_proteomics

,90430015909,90416015909,90222015909,90245015909,90450015909,90585015909,90231015909,90259015909,90290015909,90571015909,...,90251015909,90237015909,90578015909,90292015909,90248015909,90439015909,90567015909,90218015909,90406015909,90432015909
gene_symbol,,,,,,,,,,,,,,,,,,,,,
Aaas,1.599160,1.915433,1.750580,1.825331,1.834237,2.154741,1.583102,1.826309,2.006239,1.989773,...,1.930371,2.124853,1.705698,1.644431,1.898535,1.820294,1.927269,1.644101,2.128393,2.000418
Aacs,1.744862,1.937587,1.835613,1.803431,1.884205,2.254106,1.957267,1.900686,1.954829,2.102647,...,2.124660,2.268130,1.828275,1.830165,1.865827,1.756730,2.088916,1.677705,2.245752,1.884232
Aadat,2.045861,2.155587,2.223243,1.602036,2.109768,1.715294,2.168958,1.712549,2.369556,1.910213,...,2.319292,2.536160,1.449190,1.871178,1.826101,1.915133,1.612303,1.860855,1.782810,2.245230
Aars1,1.961634,1.979359,1.833880,1.952713,2.128129,2.301213,1.926340,1.789798,2.211397,2.108895,...,2.058899,2.314981,1.679398,1.786554,1.838138,1.932260,2.050858,1.729824,2.218729,1.984248
Aasdhppt,1.928826,1.904401,1.926096,1.868167,2.040966,2.171104,1.784333,1.778042,2.070739,2.023501,...,1.932585,2.101578,1.693553,1.756506,1.839763,1.815815,1.892310,1.744494,2.082072,1.971995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zdhhc3,1.662571,2.531553,1.924312,2.083756,2.081061,2.188994,1.919117,2.003042,2.393167,2.312197,...,1.902012,2.070721,1.625078,1.795889,1.730473,1.650972,1.822179,1.642153,1.939702,1.780438
Zdhhc5,1.869436,1.923390,1.721469,1.830504,1.884503,2.249343,1.793015,1.690206,2.059847,2.079038,...,2.103688,2.217281,1.707618,1.796735,1.893558,1.823907,2.016547,1.765943,2.175167,1.895243
Zdhhc6,1.687324,2.046381,1.894393,2.010046,1.990648,2.294665,1.797322,1.835998,1.943226,2.224234,...,1.831199,2.181375,2.228677,1.689687,2.017969,1.732199,2.431011,2.017060,2.929832,2.436175


In [30]:
df = pd.DataFrame({'base': 1.0,
                   'comparison': df_proteomics[df_proteomics.columns[0]].div(df_proteomics[df_proteomics.columns[1]])
                   })
res_proteomics = gc.workflows.workflow_standard(cobra_model=rmodel, mapped_genes_comparison=df['comparison'],
                                     mapped_genes_baseline=df['base'], gene_fill=1.0)

In [9]:
res_proteomics

MAM00001c    1.004497
MAM00001e    1.003650
MAM00002c    1.000296
MAM00002e    1.000295
MAM00003c    0.998806
               ...   
MAM15006x    0.979691
MAM15007r    0.993198
MAM01368r    0.993198
MAM01104x    0.999565
MAM10018x    1.030945
Length: 8458, dtype: float64

### Transcriptomics

In [37]:
df_transcriptomics

,90217015902,90218015902,90222015902,90223015902,90225015902,90227015902,90229015902,90232015902,90237015902,90239015902,...,90564015902,90567015902,90571015902,90576015902,90578015902,90581015902,90587015902,90585015902,80029995909,80028885909
gene_id,,,,,,,,,,,,,,,,,,,,,
Gad1,0.02,0.01,0.01,0.04,0.03,0.01,0.01,0.00,0.00,0.00,...,0.03,0.01,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.00
Slc26a1,42.89,45.72,46.58,44.00,42.70,45.72,46.67,45.84,45.43,48.69,...,45.13,40.53,40.82,38.64,39.15,39.73,32.83,37.81,31.82,43.69
Xpr1,14.67,17.47,14.07,14.46,14.41,16.62,17.34,13.99,15.92,17.10,...,12.67,14.07,13.99,13.90,12.77,14.15,13.11,14.05,13.96,15.08
Idua,24.77,23.98,21.74,21.64,24.56,21.12,24.46,22.08,23.83,22.06,...,22.66,26.52,26.05,26.34,25.84,25.10,29.03,27.06,30.18,21.01
Atp5me,742.04,669.23,726.47,776.53,732.96,780.19,650.94,717.30,703.97,705.98,...,673.76,619.02,696.06,694.04,741.59,698.62,642.89,740.60,717.73,697.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Mgst2,1.61,0.84,1.05,1.48,0.21,1.58,0.67,0.22,0.22,1.61,...,2.25,0.21,0.22,0.00,0.75,0.00,0.71,1.32,0.00,0.41
Aqp9,0.04,0.00,0.00,0.00,0.00,0.00,0.03,0.00,0.00,0.00,...,0.00,0.08,0.00,0.13,0.00,0.06,0.00,0.03,0.00,0.00
Cers1,0.91,0.00,1.65,0.00,0.00,1.10,0.00,1.06,0.00,1.07,...,0.00,0.00,0.00,1.53,0.00,0.00,0.00,1.66,1.48,1.30


In [38]:
df = pd.DataFrame({'base': 1.0,
                   'comparison': df_transcriptomics[df_transcriptomics.columns[0]].div(df_transcriptomics[df_transcriptomics.columns[1]])
                   })
res = gc.workflows.workflow_standard(cobra_model=rmodel, mapped_genes_comparison=df['comparison'],
                                     mapped_genes_baseline=df['base'], gene_fill=1.0)

In [39]:
res

MAM00001c    1.199605
MAM00001e    1.164219
MAM00002c    0.928400
MAM00002e    0.869250
MAM00003c    0.981850
               ...   
MAM15006x    1.086561
MAM15007r    0.938747
MAM01368r    0.938747
MAM01104x    1.030590
MAM10018x    0.931778
Length: 8458, dtype: float64